# Agents Demo: Modeling, Evaluation, and OpenAI Review

This notebook shows how to orchestrate a full Model Zoo workflow using the `ModelingAgent`, `EvaluationAgent`, and the optional `OpenAIReviewerAgent`.

This walkthrough assumes the Model Zoo already contains the `logit_marketing_v1` configuration and metrics; the agents run directly and rely on the stored metadata.

## Prerequisites
- A Model Zoo API server running locally at `http://localhost:8000`.
- Required packages installed via `pip install -e .` (from the repo root).
- Optional: set `OPENAI_API_KEY` (or pass `api_key` explicitly) to enable OpenAI-based review.

If no API key is available, the reviewer agent falls back to a heuristic summary.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve repository root so that `agents` package is importable
repo_root = Path.cwd().resolve()
if not (repo_root / 'agents').exists():
    for parent in [repo_root] + list(repo_root.parents):
        if (parent / 'agents').exists():
            repo_root = parent
            break
sys.path.insert(0, str(repo_root))

from model_zoo.utils.env import read_env_file

# Load environment variables (including OPENAI_API_KEY) from .env if available
env_path = Path(os.environ.get('MODEL_ZOO_ENV_FILE', repo_root / '.env'))
if env_path.exists():
    values = read_env_file(env_path)
    for key, value in values.items():
        os.environ.setdefault(key, value)
    print(f'Loaded environment variables from {env_path}')
else:
    print(f'Environment file not found at {env_path}.')

if not os.environ.get('OPENAI_API_KEY') or os.environ['OPENAI_API_KEY'] == 'YOUR_OPENAI_API_KEY_HERE':
    os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_API_KEY_HERE')
    print('Update OPENAI_API_KEY in the cell above or .env before enabling the reviewer agent.')



In [ ]:
try:
    import openai
    print(f'openai version: {openai.__version__}')
except ImportError:
    print('openai package not installed. Install via `pip install openai` to enable the reviewer agent.')



In [ ]:
# Configuration definition used for evaluation context
config_payload = {
    'model': {
        'id': 'logit_marketing_v1',
        'name': 'Logistic Marketing Response',
        'version': '1.0.0',
        'kind': 'library',
        'binding': {
            'module': 'custom_logit_model',
            'callable': 'train_and_classify'
        },
        'inputs': ['feature_matrix', 'labels']
    },
    'dependencies': [
        'custom-logit-model==0.1.0',
        'scikit-learn>=1.4',
        'numpy>=1.24'
    ],
    'hyperparameters': {
        'schema': [
            {'name': 'solver', 'dtype': 'string', 'default': 'lbfgs'},
            {'name': 'max_iter', 'dtype': 'int', 'default': 200}
        ],
        'values': [
            {'model_id': 'logit_marketing_v1', 'param': 'solver', 'value': 'lbfgs', 'env': 'prod'},
            {'model_id': 'logit_marketing_v1', 'param': 'max_iter', 'value': '250', 'env': 'prod'}
        ]
    },
    'metrics': [
        {
            'metric_id': 'accuracy_metric',
            'kind': 'library',
            'direction': 'higher_is_better',
            'binding': {
                'module': 'custom_logit_model.metrics',
                'callable': 'accuracy'
            }
        }
    ],
    'variables': {
        'definitions': [
            {'name': 'feature_matrix', 'source': 'feature_matrix'},
            {'name': 'labels', 'source': 'labels'}
        ]
    },
    'context': {'timeout_s': 180}
}


In [ ]:
import numpy as np
from agents import ModelingAgent, EvaluationAgent, OpenAIReviewerAgent


In [ ]:
rng = np.random.default_rng(17)
X = rng.normal(size=(200, 4))
true_coef = np.array([1.5, -2.0, 0.7, 0.5])
logits = X @ true_coef
probabilities = 1 / (1 + np.exp(-logits))
y = (probabilities > 0.5).astype(int)

data_sources = {
    'feature_matrix': X.tolist(),
    'labels': y.tolist(),
}


In [ ]:
modeling_agent = ModelingAgent(run_timeout=240)
evaluation_agent = EvaluationAgent(
    metric_thresholds={'accuracy': 0.7},
    required_artifacts=['probabilities']
)
try:
    reviewer_agent = OpenAIReviewerAgent()  # defaults to gpt-5 with new OpenAI SDK
    print('OpenAI reviewer enabled.')
except Exception as exc:
    print(f'Reviewer unavailable: {exc}')
    reviewer_agent = OpenAIReviewerAgent(enabled=False)


In [ ]:
try:
    result = modeling_agent.execute_workflow(
        data_sources=data_sources,
        env='prod',
        # params can be overridden if needed

        context={'timeout_s': 180},
        evaluation_agent=evaluation_agent,
        reviewer_agent=reviewer_agent if reviewer_agent.enabled else None,
        register=False,
        model_id='logit_marketing_v1',
        config_payload=config_payload,
    )
    result
except Exception as exc:
    print(f'Workflow failed: {exc}')




## Scenario Runs
We will run three scenarios to demonstrate how variables and parameters affect the registered model:
1. **Base scenario**: Run with full variable set and default parameters.
2. **Missing variable**: Drop a required variable to observe error handling.
3. **Parameter tweak**: Override hyperparameters and rerun.


In [ ]:
# Scenario 1: base run with all variables
scenario1_sources = data_sources
try:
    scenario1_result = modeling_agent.execute_workflow(
        data_sources=scenario1_sources,
        env='prod',
        params={'solver': 'lbfgs'},
        context={'timeout_s': 180},
        evaluation_agent=evaluation_agent,
        reviewer_agent=reviewer_agent if reviewer_agent.enabled else None,
        register=False,
        model_id='logit_marketing_v1',
        config_payload=config_payload,
    )
    scenario1_result
except Exception as exc:
    print(f'Scenario 1 failed: {exc}')


In [ ]:
# Scenario 2: remove 'labels' variable to simulate missing input
scenario2_sources = dict(data_sources)
scenario2_sources.pop('labels', None)
try:
    scenario2_result = modeling_agent.execute_workflow(
        data_sources=scenario2_sources,
        env='prod',
        params={'solver': 'lbfgs'},
        context={'timeout_s': 180},
        evaluation_agent=evaluation_agent,
        reviewer_agent=reviewer_agent if reviewer_agent.enabled else None,
        register=False,
        model_id='logit_marketing_v1',
        config_payload=config_payload,
    )
    scenario2_result
except Exception as exc:
    print(f'Scenario 2 expected failure: {exc}')


In [ ]:
# Scenario 3: tweak hyperparameters and rerun
scenario3_params = {'solver': 'liblinear', 'max_iter': 500}
try:
    scenario3_result = modeling_agent.execute_workflow(
        data_sources=data_sources,
        env='prod',
        params=scenario3_params,
        context={'timeout_s': 180},
        evaluation_agent=evaluation_agent,
        reviewer_agent=reviewer_agent if reviewer_agent.enabled else None,
        register=False,
        model_id='logit_marketing_v1',
        config_payload=config_payload,
    )
    scenario3_result
except Exception as exc:
    print(f'Scenario 3 failed: {exc}')
